In [1]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from tqdm import tqdm
import statsmodels.api as sm
from scipy import stats

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [2]:
# Subset Olink RVAT significant

olink_burden_test_file = "proteomics_prs_df_loftee_mac20_burden_regression_results.parquet"
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/blacklist/{olink_burden_test_file} -o /home/dnanexus/data_dir/

olink_whitelist = (
    pl.read_parquet(f'/home/dnanexus/data_dir/{olink_burden_test_file}')
    .rename({'gene': 'region'})
    .filter((pl.col('padj')<=0.05) & (pl.col('wilcox_padj')<=0.05) )
    .select(['region'])
    .with_columns(
        phenotype = pl.col('region') + '_olink'
    )
)

olink_correlations_file = "olink_all_mac20_lofteeHC_correlations.parquet"
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/{olink_correlations_file} -o /home/dnanexus/data_dir/

olink_corrs = (
    pl.read_parquet(f'/home/dnanexus/data_dir/{olink_correlations_file}')
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .drop_nans()
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
)

olink_whitelist = (
    olink_whitelist
    .join(olink_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
)

olink_whitelist

[===========================================================>] Completed 176,986 of 176,986 bytes (100%) /home/dnanexus/data_dir/proteomics_prs_df_loftee_mac20_burden_regression_results.parquett
Error: path
"/home/dnanexus/data_dir/olink_all_mac20_lofteeHC_correlations.parquet"
already exists but -f/--overwrite was not set


region,phenotype,loftee_corr,loftee_corr_abs,loftee_corr_dir
str,str,f64,f64,f64
"""ENSG00000163606""","""ENSG00000163606_olink""",0.204082,0.204082,1.0
"""ENSG00000144857""","""ENSG00000144857_olink""",0.162616,0.162616,1.0
"""ENSG00000161270""","""ENSG00000161270_olink""",0.175562,0.175562,1.0
"""ENSG00000091129""","""ENSG00000091129_olink""",0.165924,0.165924,1.0
"""ENSG00000132437""","""ENSG00000132437_olink""",0.214824,0.214824,1.0
…,…,…,…,…
"""ENSG00000101160""","""ENSG00000101160_olink""",0.386419,0.386419,1.0
"""ENSG00000130528""","""ENSG00000130528_olink""",0.305544,0.305544,1.0
"""ENSG00000110448""","""ENSG00000110448_olink""",0.337765,0.337765,1.0


In [3]:
# Configuration and paths
mac = 20

abs_phenotypes = False
extreme_pheno_dir = 'bottom'

eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

# Load annotation configuration
config_path = "/home/dnanexus/ukbgym/config_odds_categories.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i8
"""genomic_region""","""mane_cds""","""#E31A1C""","""MANE CDS""",1
"""genomic_region""","""non_mane_cds""","""#FB9A99""","""Other CDS""",1
"""genomic_region""","""mane_exonic""","""#33A02C""","""MANE Exon""",1
"""genomic_region""","""non_mane_exonic""","""#B2DF8A""","""Other Exon""",1
"""loftee_categories""","""loftee_hc""","""#E31A1C""","""LOFTEE HC""",1
…,…,…,…,…
"""all_encode_annotations""","""encode_ca_ctcf""","""#4DAF4A""","""Accessible Region with CTCF Bi…",1
"""all_encode_annotations""","""encode_ca""","""#377EB8""","""Chromatin Accessible""",1
"""all_encode_annotations""","""encode_ca_tf""","""#1D6498""","""Accessible Region with TF Bind…",1


In [4]:
RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "/home/dnanexus/data_dir"

ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")

variant_filter_label = "Non-CDS"  # ← change per run: "Gene body", "CDS", "Non-CDS", "Intronic", etc.

anno = (
    anno
    .with_columns(
        core_promoter = pl.col('dist_to_tss').abs() <= 50,
        encode_promoter = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_pls', 'encode_ca-h3k4me3']),
        encode_enhancer = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_pels', 'encode_dels']),
        encode_any_tf = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_tf', 'encode_ca-tf']),
        encode_annotated = pl.col('not_annotated_in_encode') == False,
        canonical_splice_variant = (pl.col('consequence_splice_acceptor_variant') == True) | (pl.col('consequence_splice_donor_variant') == True)
    )
    .filter(
        # Choose CDS
        # (pl.col('vep_cds_relaxed')==True),

        # Choose non CDS only
        # ((pl.col('vep_cds_relaxed')==False) & (pl.col('mane_cds')==False) & (pl.col('non_mane_cds')==False)),

        # Choose Gene Body
        # ((pl.col('consequence_upstream_gene_variant') == False) & (pl.col('consequence_downstream_gene_variant') == False)),

        # Choose VEP consequence
        # (pl.col('consequence_missense_variant') == True),
        # (pl.any_horizontal(cs.contains("consequence_splice") == True)),

        # Core promoter
        # (pl.col('promoterai_is_na') == False),

        # Choose Introns with/without alternate CDS
        # (pl.col('consequence_intron_variant') == True),

        # Choose regulatory region
        # (pl.col('not_in_encode') == False),
        # (pl.col('encode_eh_pr') == True),
        # (pl.col('encode_all_tf') == True),

        # Filter to olink whitelist regions
        (pl.col('region').is_in(olink_whitelist['region'].unique()))
    )
    
    .with_columns(
        loftee_disorder = pl.all_horizontal((pl.col(c) == True) for c in ['loftee_hc', 'mobi_curated_disorder_priority']),
        loftee_lip = pl.all_horizontal((pl.col(c) == True) for c in ['loftee_hc', 'mobi_lip_full']),
        loftee_ted = pl.all_horizontal((pl.col(c) == True) for c in ['loftee_hc', 'ted_domain']),
        loftee_low_complexity = pl.all_horizontal((pl.col(c) == True) for c in ['loftee_hc', 'low_complexity_domain']),
    )
    .with_columns(
        indel_length = (pl.col('ref').str.len_chars() - pl.col('alt').str.len_chars()).abs(),
        insertion = pl.when(pl.col('ref').str.len_chars() < pl.col('alt').str.len_chars()).then(True).otherwise(False),
        deletion = pl.when(pl.col('ref').str.len_chars() > pl.col('alt').str.len_chars()).then(True).otherwise(False),
    )
    .with_columns(
        # --- Indel CDS categories ---
        deletion_gt2bp = pl.when(pl.col('deletion') & (pl.col('indel_length') >= 2)).then(True).otherwise(False),
        deletion_gt5bp = pl.when(pl.col('deletion') & (pl.col('indel_length') >= 5)).then(True).otherwise(False),
        insertion_gt2bp = pl.when(pl.col('insertion') & (pl.col('indel_length') >= 2)).then(True).otherwise(False),
        insertion_gt5bp = pl.when(pl.col('insertion') & (pl.col('indel_length') >= 5)).then(True).otherwise(False),
        deletion_x3bp = pl.when(pl.col('deletion') & (pl.col('indel_length') %3 == 0)).then(True).otherwise(False),
        deletion_not_x3bp = pl.when(pl.col('deletion') & (pl.col('indel_length') %3 != 0)).then(True).otherwise(False),
        insertion_x3bp = pl.when(pl.col('insertion') & (pl.col('indel_length') %3 == 0)).then(True).otherwise(False),
        insertion_not_x3bp = pl.when(pl.col('insertion') & (pl.col('indel_length') %3 != 0)).then(True).otherwise(False),
    )
    .with_columns(
        # --- ENCODE indels ---
        encode_deletion = pl.when(pl.col('deletion') & (pl.col('encode_annotated')==True)).then(True).otherwise(False),
        encode_insertion = pl.when(pl.col('insertion') & (pl.col('encode_annotated')==True)).then(True).otherwise(False),
        encode_snp = pl.when((pl.col('encode_annotated')==True) & (pl.col('deletion')==False) & (pl.col('insertion')==False)).then(True).otherwise(False),

        # --- ENCODE deletions ---
        encode_deletion_gt2bp = pl.when((pl.col('indel_length') >= 2) & (pl.col('encode_annotated')==True)).then(True).otherwise(False) & pl.col('deletion'),
        encode_deletion_gt5bp = pl.when((pl.col('indel_length') >= 5) & (pl.col('encode_annotated')==True)).then(True).otherwise(False) & pl.col('deletion'),
    )
)


# selected_categories = ['genomic_region'] # genomic_region
# selected_categories = ['plof_consequences'] # plof consequences
# selected_categories = ['loftee_categories'] # loftee consequences
selected_categories = ['plof_consequences', 'missense'] # CDS
# selected_categories = ['protein_domains'] # Protein Domains
# selected_categories = ['loftee_protein_domains'] # LOFTEE Protein Domains
# selected_categories = ['splicing'] # Splicing
# selected_categories = ['non_coding_regions'] # Non-coding regions
# selected_categories = ['in_or_del'] # encode
# selected_categories = ['in_or_del', 'indel_lengths'] # indel length
# selected_categories = ['indel_cds'] # CDS indels
# selected_categories = ['encode_indels'] # encode indels
# selected_categories = ['encode_deletions'] # encode deletions
# selected_categories = ['all_encode_annotations'] # encode regions
# selected_categories = ['encode_boolean'] # encode boolean
# selected_categories = ['5utr_annotator'] # 5utr

selected_annos = anno_config_df.filter(
    pl.col('category').is_in(selected_categories)
)['annotation'].to_list()

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]
selected_annos = list(set(selected_annos).intersection(set(existing_annos)))
fillna_cols = [c+'_is_na' for c in selected_annos]

anno = (
    anno
    .select(
        set(['id', 'region']).union(set(selected_annos))
    )
    .collect(engine='streaming')
)

anno

[===========================================================>] Completed 7,370,308,270 of 7,370,308,270 bytes (100%) /home/dnanexus/data_dir/annotations_fillna_ukbgym_with_mane.parquett


/tmp/ipykernel_25457/1886995904.py:106: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
/tmp/ipykernel_25457/1886995904.py:115: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


consequence_start_lost,consequence_splice_acceptor_variant,consequence_stop_gained,consequence_stop_lost,consequence_frameshift_variant,consequence_splice_donor_variant,consequence_missense_variant,id,region
i8,i8,i8,i8,i8,i8,i8,str,str
0,0,0,0,0,0,0,"""chr1:196922428:T:A""","""ENSG00000134365"""
0,0,0,0,0,0,0,"""chr2:178785956:G:A""","""ENSG00000155657"""
0,0,0,0,0,0,0,"""chr3:140124263:A:G""","""ENSG00000158258"""
0,0,0,0,0,0,0,"""chr10:123008470:C:G""","""ENSG00000196177"""
0,0,0,0,0,0,0,"""chr4:119523359:G:A""","""ENSG00000138735"""
…,…,…,…,…,…,…,…,…
0,0,0,0,0,0,0,"""chr6:16267482:A:C""","""ENSG00000137198"""
0,0,0,0,0,0,0,"""chr8:16560986:C:A""","""ENSG00000038945"""
0,0,0,0,0,0,0,"""chr2:118973177:A:G""","""ENSG00000019169"""


In [5]:
# Handle fillna annotations
anno_fillna = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")
fillna_cols = set([c+'_is_na' for c in selected_annos]).intersection(set(anno_fillna.collect_schema().names()))

anno_fillna_melted = (
    anno_fillna
    .filter(pl.col('region').is_in(olink_whitelist['region'].unique()))
    .select(['id', 'region'] + list(fillna_cols))
    .join(
        anno.select(['id', 'region']).lazy(), 
        on=['id', 'region'], 
        how='semi'
    )
    .unpivot(
        index=["id", "region"],
        on=list(fillna_cols),
        variable_name="annotation",
        value_name="annotation_is_na"
    )
    .filter(
        pl.col('annotation_is_na') == 1
    )
    .with_columns(
        annotation = pl.col('annotation').str.replace('_is_na$', '')
    )
    # .collect(engine='streaming')
)

melted_anno = (
    anno.lazy()

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )

    # Merge with annotation configuration to get direction and filter
    .join(
        anno_config_df.select(["annotation", "category", "annotation_dir"]).lazy(),
        on="annotation",
        how="left"
    )
    .filter(pl.col('category').is_in(selected_categories))
    .unique()

    .with_columns(
        # Changed because we have binary annotations
        annotation_score_dircor = (
            pl.when(pl.col("annotation_dir")!=1)
            .then((pl.col('annotation_score')-1).abs())
            .otherwise(pl.col('annotation_score'))
        )
    )
    
    # Keep variants that don't have fillna annotation
    .join(
        anno_fillna_melted,
        on=['id', 'region', 'annotation'],
        how='anti'
    )
    # .drop_nulls()
    # .filter(pl.col('annotation_score').is_not_null())

    .collect(engine='streaming')
)

melted_anno

/tmp/ipykernel_25457/3539409218.py:70: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


id,region,annotation,annotation_score,category,annotation_dir,annotation_score_dircor
str,str,str,f32,str,i8,f32
"""chr12:95504493:TTTTTG:T""","""ENSG00000111142""","""consequence_stop_gained""",0.0,"""plof_consequences""",1,0.0
"""chr5:137031308:C:A""","""ENSG00000152377""","""consequence_stop_gained""",0.0,"""plof_consequences""",1,0.0
"""chr11:122842327:G:T""","""ENSG00000109943""","""consequence_stop_gained""",0.0,"""plof_consequences""",1,0.0
"""chr2:114889657:G:A""","""ENSG00000175497""","""consequence_stop_gained""",0.0,"""plof_consequences""",1,0.0
"""chr12:70859910:C:T""","""ENSG00000153233""","""consequence_stop_gained""",0.0,"""plof_consequences""",1,0.0
…,…,…,…,…,…,…
"""chr2:105840586:C:G""","""ENSG00000071051""","""consequence_stop_gained""",0.0,"""plof_consequences""",1,0.0
"""chr2:137970188:T:C""","""ENSG00000150540""","""consequence_stop_gained""",0.0,"""plof_consequences""",1,0.0
"""chr10:123774998:A:T""","""ENSG00000121898""","""consequence_stop_gained""",0.0,"""plof_consequences""",1,0.0


In [6]:
RAP_APPV_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var"
LOCAL_APPV_DIR = "/home/dnanexus"

# APPV_FILE = "olink_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_small_desc.parquet"
APPV_FILE = "olink_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_small_desc.parquet"

!dx download {RAP_APPV_DIR}/{APPV_FILE} -o {LOCAL_APPV_DIR}/{APPV_FILE}
olink_appv = pl.scan_parquet(f"{LOCAL_APPV_DIR}/{APPV_FILE}")

# Create a lazy frame with the unique keys
anno_keys = anno.select(pl.col('id').unique()).lazy()

# Merge phenotype data and annotation data
olink_appv = (
    olink_appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        (pl.col('n_individuals') <= mac)
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value', 'n_individuals']
    )
)

[===========================================================>] Completed 100,324,032 of 100,324,032 bytes (100%) /home/dnanexus/olink_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_small_desc.parquett


In [7]:
n_steps = 101
or_threshold_pheno_left = 1
or_threshold_pheno_right = 1e-5

cutoffs_list = 1 - np.logspace(
    np.log10(or_threshold_pheno_left),
    np.log10(or_threshold_pheno_right),
    num=n_steps
)

# Cast to float32 first, then add 0.99 as float32
cutoffs_f32 = cutoffs_list.astype(np.float32)
cutoffs_f32 = np.append(cutoffs_f32, np.float32(0.99))

# Prepare cutoffs sorted and deduplicated
cutoffs_sorted = (
    pl.LazyFrame({"pheno_cutoff": cutoffs_f32.tolist()})
    .unique()
    .sort('pheno_cutoff')
    .collect()
)

# Convert percentile cutoffs to z-score thresholds (assuming standard Gaussian)
cutoffs_sorted = cutoffs_sorted.with_columns(
    zscore_cutoff = pl.Series('zscore_cutoff', stats.norm.ppf(cutoffs_sorted['pheno_cutoff'].to_numpy()))
)

print(f"Total cutoffs: {cutoffs_sorted.shape[0]}")
print(f"0.99 included (within float32 precision): {any(abs(x - 0.99) < 1e-6 for x in cutoffs_sorted['pheno_cutoff'].to_list())}")
cutoffs_sorted

Total cutoffs: 101
0.99 included (within float32 precision): True


pheno_cutoff,zscore_cutoff
f64,f64
0.0,-inf
0.108749,-1.233208
0.205672,-0.821532
0.292054,-0.547394
0.369043,-0.33439
…,…
0.999984,4.160827
0.999986,4.187112
0.999987,4.213421


## Odds ratio computation

In [ ]:
# --- OR computation with z-score thresholds + bootstrap CIs ---

id_region = anno.select(['id', 'region']).unique().lazy()

# Stream-collect base data ONCE: direction-corrected z-score
gp_base = (
    olink_appv
    .join(id_region, on='id', how='inner')
    .join(olink_whitelist.lazy(), on=['region', 'phenotype'], how='inner')
    .with_columns(
        mean_pheno_value_dircor = pl.when(pl.col('loftee_corr_dir') == -1)
            .then(-pl.col('mean_pheno_value'))
            .otherwise(pl.col('mean_pheno_value'))
    )
    .select(['id', 'region', 'mean_pheno_value_dircor'])
    .collect(engine='streaming')
)

print(f"gp_base: {gp_base.shape}")

# --- Step 1: Pre-aggregate per (region, annotation, cutoff) 2x2 table counts ---
all_regions = sorted(olink_whitelist['region'].unique().to_list())
n_regions = len(all_regions)
all_regions_df = pl.DataFrame({'region': all_regions})

annotations_list = sorted(melted_anno['annotation'].unique().to_list())
n_annotations = len(annotations_list)
n_cutoffs = cutoffs_sorted.shape[0]

counts_4d = np.zeros((n_annotations, n_regions, n_cutoffs, 4), dtype=np.int64)

for a_idx, annotation in enumerate(tqdm(annotations_list, desc='Pre-aggregating')):
    anno_data = (
        melted_anno
        .filter(pl.col('annotation') == annotation)
        .select(['id', 'region', 'annotation_score_dircor'])
    )

    region_zscore_counts = (
        gp_base.lazy()
        .join(anno_data.lazy(), on=['id', 'region'], how='inner')
        .group_by(['region', 'mean_pheno_value_dircor'])
        .agg(
            n_annotated = (pl.col('annotation_score_dircor') == 1).sum().cast(pl.Int64),
            n_not_annotated = (pl.col('annotation_score_dircor') == 0).sum().cast(pl.Int64),
        )
        .collect()
        .sort(['region', 'mean_pheno_value_dircor'])
        .with_columns(
            cum_annotated = pl.col('n_annotated').cum_sum().over('region'),
            cum_not_annotated = pl.col('n_not_annotated').cum_sum().over('region'),
        )
    )

    region_totals = (
        region_zscore_counts
        .group_by('region')
        .agg(
            total_annotated = pl.col('n_annotated').sum(),
            total_not_annotated = pl.col('n_not_annotated').sum(),
        )
    )

    per_region = (
        cutoffs_sorted
        .join(all_regions_df, how='cross')
        .sort(['region', 'zscore_cutoff'])
        .join_asof(
            region_zscore_counts.sort(['region', 'mean_pheno_value_dircor']),
            left_on='zscore_cutoff',
            right_on='mean_pheno_value_dircor',
            by='region',
            strategy='backward',
        )
        .with_columns(
            cum_annotated = pl.col('cum_annotated').fill_null(0),
            cum_not_annotated = pl.col('cum_not_annotated').fill_null(0),
        )
        .join(region_totals, on='region', how='left')
        .with_columns(
            total_annotated = pl.col('total_annotated').fill_null(0),
            total_not_annotated = pl.col('total_not_annotated').fill_null(0),
        )
        .with_columns(
            n_dis_above_cutoff = pl.col('total_annotated') - pl.col('cum_annotated'),
            n_notdis_above_cutoff = pl.col('cum_annotated'),
            n_dis_below_cutoff = pl.col('total_not_annotated') - pl.col('cum_not_annotated'),
            n_notdis_below_cutoff = pl.col('cum_not_annotated'),
        )
        .sort(['region', 'zscore_cutoff'])
    )

    counts_4d[a_idx, :, :, 0] = per_region['n_dis_above_cutoff'].to_numpy().reshape(n_regions, n_cutoffs)
    counts_4d[a_idx, :, :, 1] = per_region['n_notdis_above_cutoff'].to_numpy().reshape(n_regions, n_cutoffs)
    counts_4d[a_idx, :, :, 2] = per_region['n_dis_below_cutoff'].to_numpy().reshape(n_regions, n_cutoffs)
    counts_4d[a_idx, :, :, 3] = per_region['n_notdis_below_cutoff'].to_numpy().reshape(n_regions, n_cutoffs)

print(f"Pre-aggregated: {counts_4d.shape} = {n_annotations} annos × {n_regions} regions × {n_cutoffs} cutoffs × 4 counts")

# --- Step 2: Point estimate ---
total_counts = counts_4d.sum(axis=1)
with np.errstate(divide='ignore', invalid='ignore'):
    point_or = (total_counts[:, :, 0] / total_counts[:, :, 1]) / \
               (total_counts[:, :, 2] / total_counts[:, :, 3])

# --- Step 3: Bootstrap CIs ---
n_boot = 1000
rng = np.random.default_rng(42)
boot_ors = np.full((n_boot, n_annotations, n_cutoffs), np.nan)

for b in tqdm(range(n_boot), desc='Bootstrapping'):
    idx = rng.choice(n_regions, size=n_regions, replace=True)
    sampled = counts_4d[:, idx, :, :].sum(axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        boot_ors[b] = (sampled[:, :, 0] / sampled[:, :, 1]) / (sampled[:, :, 2] / sampled[:, :, 3])

boot_ci_lower = np.nanpercentile(boot_ors, 2.5, axis=0)
boot_ci_upper = np.nanpercentile(boot_ors, 97.5, axis=0)

# --- Build output DataFrame ---
cutoff_values = cutoffs_sorted['pheno_cutoff'].to_numpy()
zscore_values = cutoffs_sorted['zscore_cutoff'].to_numpy()
anno_idx, cutoff_idx = np.meshgrid(np.arange(n_annotations), np.arange(n_cutoffs), indexing='ij')

or_df = (
    pl.DataFrame({
        'annotation': np.array(annotations_list)[anno_idx.ravel()],
        'pheno_cutoff': cutoff_values[cutoff_idx.ravel()],
        'zscore_cutoff': zscore_values[cutoff_idx.ravel()],
        'n_dis_above_cutoff': total_counts[:, :, 0].ravel(),
        'n_notdis_above_cutoff': total_counts[:, :, 1].ravel(),
        'n_dis_below_cutoff': total_counts[:, :, 2].ravel(),
        'n_notdis_below_cutoff': total_counts[:, :, 3].ravel(),
        'odds_ratio': point_or.ravel(),
        'ci_lower': boot_ci_lower.ravel(),
        'ci_upper': boot_ci_upper.ravel(),
    })
    .filter(pl.col('odds_ratio').is_finite() & (pl.col('odds_ratio') > 0))
    .sort('pheno_cutoff')
)

print('unique percentile cutoffs:', or_df['pheno_cutoff'].n_unique())
or_df

gp_base: (4988993, 3)


Pre-aggregating: 100%|██████████| 7/7 [00:23<00:00,  3.39s/it]


Pre-aggregated: (7, 1119, 101, 4) = 7 annos × 1119 regions × 101 cutoffs × 4 counts


Bootstrapping: 100%|██████████| 1000/1000 [00:05<00:00, 178.00it/s]
/home/dnanexus/deeprvat2-env/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered


annotation,pheno_cutoff,zscore_cutoff,n_dis_above_cutoff,n_notdis_above_cutoff,n_dis_below_cutoff,n_notdis_below_cutoff,odds_ratio,ci_lower,ci_upper
str,f64,f64,i64,i64,i64,i64,f64,f64,f64
"""consequence_frameshift_variant""",0.108749,-1.233208,1004,2986,4497217,487786,0.036469,0.029632,0.045478
"""consequence_missense_variant""",0.108749,-1.233208,58974,21698,4439247,469074,0.287192,0.263753,0.318289
"""consequence_splice_acceptor_va…",0.108749,-1.233208,237,638,4497984,490134,0.040479,0.032329,0.05006
"""consequence_splice_donor_varia…",0.108749,-1.233208,363,892,4497858,489880,0.044323,0.037557,0.052833
"""consequence_start_lost""",0.108749,-1.233208,68,184,4498153,490588,0.040306,0.029583,0.053747
…,…,…,…,…,…,…,…,…,…
"""consequence_splice_acceptor_va…",0.99999,4.264588,3,872,824,4987294,20.822963,0.0,48.711549
"""consequence_splice_donor_varia…",0.99999,4.264588,10,1245,817,4986921,49.02765,10.031429,105.443951
"""consequence_start_lost""",0.99999,4.264588,1,251,826,4987915,24.058319,0.0,80.801428


## Plotting

In [ ]:
plt_df = (
    or_df
    .drop_nans()
    .filter(
        pl.col('odds_ratio').is_finite(),
        pl.col('odds_ratio') > 0,
        pl.col('ci_lower') > 0,
        # pl.col('n_dis_above_cutoff')>1,
        # pl.col('n_dis_below_cutoff')>1,
        # pl.col('n_notdis_above_cutoff')>1,
        # pl.col('n_notdis_below_cutoff')>1,
    )

    .with_columns(
        pheno_cutoff = pl.col('pheno_cutoff').cast(pl.Float32),
        pheno_cutoff_inv = 1 - pl.col('pheno_cutoff'),
    )
    .with_columns(
        neg_log_pheno_cutoff_inv = -np.log10(pl.col('pheno_cutoff_inv')),
    )

    # Join plotting info
    .join(
        anno_config_df.filter(pl.col('category').is_in(selected_categories)).select(['annotation', 'color', 'label']).unique(),
        on='annotation',
        how='left'
    )
    
    .sort('pheno_cutoff_inv')
)

plt_df

unique percentile cutoffs: 100


annotation,pheno_cutoff,zscore_cutoff,n_dis_above_cutoff,n_notdis_above_cutoff,n_dis_below_cutoff,n_notdis_below_cutoff,odds_ratio,ci_lower,ci_upper,pheno_cutoff_inv,neg_log_pheno_cutoff_inv,color,label
str,f32,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,str,str
"""consequence_frameshift_variant""",0.99999,4.264588,8,3982,819,4984184,12.226408,4.231448,21.570779,0.00001,4.999411,"""#E31A1C""","""VEP Frameshift"""
"""consequence_missense_variant""",0.99999,4.264588,90,80582,737,4907584,7.437119,5.149497,10.066776,0.00001,4.999411,"""#1E90FF""","""VEP Missense"""
"""consequence_splice_donor_varia…",0.99999,4.264588,10,1245,817,4986921,49.02765,10.031429,105.443951,0.00001,4.999411,"""#6A1B9A""","""VEP Splice Donor"""
"""consequence_stop_gained""",0.99999,4.264588,8,3040,819,4985126,16.018013,3.876341,30.872128,0.00001,4.999411,"""#FB9A99""","""VEP Stop Gained"""
"""consequence_frameshift_variant""",0.999989,4.239405,8,3982,863,4984140,11.602943,4.058546,20.553136,0.000011,4.950562,"""#E31A1C""","""VEP Frameshift"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""consequence_splice_acceptor_va…",0.108749,-1.233208,237,638,4497984,490134,0.040479,0.032329,0.05006,0.891251,0.05,"""#AB47BC""","""VEP Splice Acceptor"""
"""consequence_splice_donor_varia…",0.108749,-1.233208,363,892,4497858,489880,0.044323,0.037557,0.052833,0.891251,0.05,"""#6A1B9A""","""VEP Splice Donor"""
"""consequence_start_lost""",0.108749,-1.233208,68,184,4498153,490588,0.040306,0.029583,0.053747,0.891251,0.05,"""#E65100""","""VEP Start Lost"""


In [ ]:
plot_df = (
    plt_df
    .filter(
        # (~pl.col('annotation').is_in(['not_annotated_in_encode', 'encode_annotated']))
    )
)

# Generate breaks and labels for x-axis
start_exp = int(-np.log10(or_threshold_pheno_left))
end_exp = int(-np.log10(or_threshold_pheno_right))
breaks_linear = np.arange(start_exp, end_exp + 1)
labels_sci = [f"$10^{{-{int(b)}}}$" for b in breaks_linear]

color_dict = dict(zip(plt_df['label'], plt_df['color']))
pheno_dir = 'underexpression' if extreme_pheno_dir.lower() == 'bottom' else 'overexpression'

(
    ggplot(
        plot_df,
        aes(x='neg_log_pheno_cutoff_inv', y='odds_ratio')
    )
    + geom_hline(aes(yintercept=1), color='black', linetype='dotted')
    + geom_line(aes(color='label'), size=1)
    + geom_ribbon(aes(ymin='ci_lower', ymax='ci_upper', fill='label'), alpha=0.1)
    + scale_fill_manual(values=color_dict)
    + scale_color_manual(values=color_dict)
    + labs(
        title=f"Olink {olink_whitelist.shape[0]} proteins",
        y=f"Odds Ratio of being\nin an Annotated Region",
        x=f"Extreme Phenotype Quantile ({pheno_dir})",
        color="Annotation",
        fill="Annotation",
    )
    + scale_x_continuous(
        breaks=breaks_linear,
        labels=labels_sci
    )
    + scale_y_log10()
    + annotation_logticks(sides="lb")
    + theme_minimal()
    + theme(
        figure_size=(8, 6),
        axis_text=element_text(size=13),
        axis_title=element_text(size=13, lineheight=1.4),
        legend_text=element_text(size=13),
        legend_title=element_text(size=13),
        legend_position=(0.1, 0.9), 
        legend_background=element_rect(fill="white", color='white', alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)

In [11]:
plt_df.filter(pl.col('pheno_cutoff') == 0.99)

annotation,pheno_cutoff,zscore_cutoff,n_dis_above_cutoff,n_notdis_above_cutoff,n_dis_below_cutoff,n_notdis_below_cutoff,odds_ratio,ci_lower,ci_upper,pheno_cutoff_inv,neg_log_pheno_cutoff_inv,color,label
str,f32,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,str,str
"""consequence_frameshift_variant""",0.99,2.326348,56,3934,37063,4947940,1.900367,1.276864,2.61672,0.01,2.0,"""#E31A1C""","""VEP Frameshift"""
"""consequence_missense_variant""",0.99,2.326348,1318,79354,35801,4872520,2.260503,2.01321,2.521806,0.01,2.0,"""#1E90FF""","""VEP Missense"""
"""consequence_splice_acceptor_va…",0.99,2.326348,8,867,37111,4951007,1.231011,0.469123,2.196967,0.01,2.0,"""#AB47BC""","""VEP Splice Acceptor"""
"""consequence_splice_donor_varia…",0.99,2.326348,23,1232,37096,4950642,2.491447,1.308199,3.911524,0.01,2.0,"""#6A1B9A""","""VEP Splice Donor"""
"""consequence_stop_gained""",0.99,2.326348,40,3008,37079,4948866,1.774843,1.143972,2.450276,0.01,2.0,"""#FB9A99""","""VEP Stop Gained"""


In [12]:
# --- Save reference OR points for reuse in continuous annotation notebooks ---
# Set this label to describe the variant filter used in this run
# variant_filter_label = "Gene body"   # ← change per run: "Gene body", "CDS", "Non-CDS", "Intronic", etc.

RAP_REF_OR = 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/odds_ratio_results'
ref_or_local = f'{LOCAL_APPV_DIR}/olink_category_or_reference_points.tsv'

# Count annotated variants (score == 1) per annotation
n_annotated_per_anno = (
    melted_anno
    .filter(pl.col('annotation_score_dircor') == 1)
    .group_by('annotation')
    .agg(n_variants = pl.len().cast(pl.Int64))
)

# Tag this run's results with variant filter label and per-annotation variant count
run_results = (
    plt_df
    .filter(pl.col('pheno_cutoff') == 0.99)
    .with_columns(variant_filter = pl.lit(variant_filter_label))
    .join(n_annotated_per_anno, on='annotation', how='left')
)

# Try to download existing file; if it doesn't exist, start fresh
!dx download {RAP_REF_OR}/olink_category_or_reference_points.tsv -o {ref_or_local} -f 2>/dev/null || echo "No existing file, starting fresh"

import os
if os.path.exists(ref_or_local):
    existing = pl.read_csv(ref_or_local, separator='\t')
    # Remove stale entries for same (variant_filter, annotation)
    new_keys = run_results.select(['variant_filter', 'annotation']).unique()
    existing = existing.join(new_keys, on=['variant_filter', 'annotation'], how='anti')
    combined = pl.concat([existing, run_results], how='vertical_relaxed')
    print(f"Appended to existing file ({existing.shape[0]} old + {run_results.shape[0]} new)")
else:
    combined = run_results
    print(f"Starting fresh file ({run_results.shape[0]} rows)")

combined.write_csv(ref_or_local, separator='\t')

# Upload to DNAnexus
!dx mkdir -p {RAP_REF_OR}
!dx rm {RAP_REF_OR}/olink_category_or_reference_points.tsv -f 2>/dev/null || echo "No existing file to remove, proceeding to upload"
!dx upload {ref_or_local} --path {RAP_REF_OR}/

No existing file, starting fresh
Starting fresh file (5 rows)
Could not resolve "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/odd
s_ratio_results/olink_category_or_reference_points.tsv": Unable to resolve
"olink_category_or_reference_points.tsv" to a data object or folder name in
'/processed_data/ukbgym/odds_ratio_results'
No existing file to remove, proceeding to upload
[===========================================================>] Uploaded 1,285 of 1,285 bytes (100%) /home/dnanexus/olink_category_or_reference_points.tsv
ID                                file-J68K2FjJg0yPGy57z9jyF3J8
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/ukbgym/odds_ratio_results
Name                              olink_category_or_reference_points.tsv
State                             closing
Visibility                        visible
Types                             -
Properties            

In [13]:
print(f"\nSaved {combined.shape[0]} reference points to {RAP_REF_OR}/olink_category_or_reference_points.tsv")
print(f"Variant filters: {sorted(combined['variant_filter'].unique().to_list())}")
print(f"Annotations: {sorted(combined['annotation'].unique().to_list())}")
print(f"\nAnnotated variant counts for this run:")
print(n_annotated_per_anno.sort('annotation'))


Saved 5 reference points to project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/odds_ratio_results/olink_category_or_reference_points.tsv
Variant filters: ['Non-CDS']
Annotations: ['consequence_frameshift_variant', 'consequence_missense_variant', 'consequence_splice_acceptor_variant', 'consequence_splice_donor_variant', 'consequence_stop_gained']

Annotated variant counts for this run:
shape: (7, 2)
┌─────────────────────────────────┬────────────┐
│ annotation                      ┆ n_variants │
│ ---                             ┆ ---        │
│ str                             ┆ i64        │
╞═════════════════════════════════╪════════════╡
│ consequence_frameshift_variant  ┆ 21132      │
│ consequence_missense_variant    ┆ 363268     │
│ consequence_splice_acceptor_va… ┆ 4513       │
│ consequence_splice_donor_varia… ┆ 6084       │
│ consequence_start_lost          ┆ 1116       │
│ consequence_stop_gained         ┆ 14033      │
│ consequence_stop_lost           ┆ 462        │
└───